In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, HTML, Layout, GridBox
from IPython.display import display

# ============================================================
# THERMAL NOISE THROUGH AN RC FILTER
# FAST WIDGET VERSION
# ============================================================

plt.ioff()

# ============================================================
# CONSTANTS
# ============================================================

kB = 1.380649e-23

# Frequency axis is created only once
f = np.logspace(0, 6, 1200)
omega = 2.0 * np.pi * f

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#8b4d13;
    margin-bottom:8px;
">
Thermal Noise Through an RC Low-Pass Filter
</div>

<div style="margin-bottom:4px;">
A resistor at temperature T produces approximately white thermal noise with PSD Sᵥᵥ = 2kTR.
</div>

<div style="margin-bottom:4px;">
An RC low-pass filter shapes this spectrum according to |H(jω)|² = 1/(1 + ω²R²C²).
</div>

<div style="margin-bottom:4px;">
The output noise power is finite and satisfies Rᵧᵧ(0) = kT/C.
</div>

<div>
<b>This notebook:</b> shows how R, C and temperature affect the spectral shape and total thermal-noise power.
</div>

</div>
""")

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}
slider_layout = Layout(width='160px')

R_slider = FloatSlider(
    min=1.0,
    max=20.0,
    step=1.0,
    value=10.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

C_slider = FloatSlider(
    min=1.0,
    max=100.0,
    step=1.0,
    value=20.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

T_slider = FloatSlider(
    min=250.0,
    max=400.0,
    step=10.0,
    value=300.0,
    description=' ',
    continuous_update=True,
    readout=False,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# CURRENT VALUE LABELS
# ============================================================

R_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">10 kΩ</div>'
)

C_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">20 nF</div>'
)

T_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">300 K</div>'
)

# ============================================================
# CONTROL LABELS
# ============================================================

R_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">Resistance R:</div>'
)

C_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">Capacitance C:</div>'
)

T_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">Temperature T:</div>'
)

# ============================================================
# CONTROL PANEL
# ============================================================

controls_grid = GridBox(
    children=[
        R_label, R_slider, R_value,
        C_label, C_slider, C_value,
        T_label, T_slider, T_value
    ],
    layout=Layout(
        width='770px',
        grid_template_columns='120px 160px 75px 120px 160px 75px',
        grid_template_rows='34px 34px',
        grid_gap='5px 8px',
        align_items='center',
        overflow='hidden'
    )
)

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#8b4d13;
            margin-bottom:5px;
        ">
        Circuit Parameters
        </div>
        """),
        controls_grid
    ],
    layout=Layout(
        width='800px',
        padding='10px 14px',
        border='1px solid #ddc7af',
        margin='10px 0px 8px 0px',
        overflow='hidden'
    )
)

# ============================================================
# INITIAL PARAMETER VALUES
# ============================================================

R0 = R_slider.value * 1e3
C0 = C_slider.value * 1e-9
T0 = T_slider.value

S_in0 = 2.0 * kB * T0 * R0

H20 = 1.0 / (1.0 + (omega * R0 * C0)**2)

S_out0 = S_in0 * H20

fc0 = 1.0 / (2.0 * np.pi * R0 * C0)

cumulative0 = (2.0 / np.pi) * np.arctan(omega * R0 * C0)

# ============================================================
# FIGURE 1
# THERMAL-NOISE INPUT PSD
# ============================================================

fig1, ax1 = plt.subplots(figsize=(5.1, 3.4))

line_input, = ax1.plot(
    f,
    S_in0 * np.ones_like(f),
    linewidth=2.0
)

ax1.set_xscale('log')
ax1.set_yscale('log')

ax1.set_xlim(
    1,
    1e6
)

ax1.set_ylim(
    1e-19,
    1e-15
)

ax1.set_xlabel(
    'Frequency f (Hz)',
    fontsize=11
)

ax1.set_ylabel(
    'PSD (V² s)',
    fontsize=11
)

ax1.set_title(
    'Thermal-Noise Input PSD',
    fontsize=13,
    pad=9
)

ax1.tick_params(
    axis='both',
    labelsize=9
)

ax1.grid(
    True,
    which='major',
    linestyle=':',
    alpha=0.5
)

fig1.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.88,
    bottom=0.18
)

# ============================================================
# FIGURE 2
# OUTPUT NOISE PSD
# ============================================================

fig2, ax2 = plt.subplots(figsize=(5.1, 3.4))

line_output, = ax2.plot(
    f,
    S_out0,
    linewidth=2.0
)

cutoff_line = ax2.axvline(
    fc0,
    linestyle='--',
    linewidth=1.2,
    label='Cutoff frequency'
)

ax2.set_xscale('log')
ax2.set_yscale('log')

ax2.set_xlim(
    1,
    1e6
)

ax2.set_ylim(
    1e-22,
    1e-15
)

ax2.set_xlabel(
    'Frequency f (Hz)',
    fontsize=11
)

ax2.set_ylabel(
    'PSD (V² s)',
    fontsize=11
)

ax2.set_title(
    'Output Noise PSD',
    fontsize=13,
    pad=9
)

ax2.tick_params(
    axis='both',
    labelsize=9
)

ax2.grid(
    True,
    which='major',
    linestyle=':',
    alpha=0.5
)

ax2.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.18),
    fontsize=8
)

fig2.subplots_adjust(
    left=0.15,
    right=0.97,
    top=0.88,
    bottom=0.25
)

# ============================================================
# FIGURE 3
# ACCUMULATED OUTPUT NOISE POWER
# ============================================================

fig3, ax3 = plt.subplots(figsize=(10.4, 3.5))

line_cumulative, = ax3.plot(
    f,
    cumulative0,
    linewidth=2.0
)

cutoff_line_3 = ax3.axvline(
    fc0,
    linestyle='--',
    linewidth=1.2
)

ax3.set_xscale('log')

ax3.set_xlim(
    1,
    1e6
)

ax3.set_ylim(
    0,
    1.05
)

ax3.set_xlabel(
    'Frequency f (Hz)',
    fontsize=11
)

ax3.set_ylabel(
    'Fraction of total output noise power',
    fontsize=11
)

ax3.set_title(
    'Accumulated Output Noise Power',
    fontsize=13,
    pad=9
)

ax3.tick_params(
    axis='both',
    labelsize=9
)

ax3.grid(
    True,
    which='major',
    linestyle=':',
    alpha=0.5
)

fig3.subplots_adjust(
    left=0.09,
    right=0.97,
    top=0.87,
    bottom=0.18
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_plots(change=None):

    # --------------------------------------------------------
    # CURRENT PARAMETERS
    # --------------------------------------------------------

    R_kohm = R_slider.value
    C_nF = C_slider.value
    T = T_slider.value

    R = R_kohm * 1e3
    C = C_nF * 1e-9

    # --------------------------------------------------------
    # CURRENT VALUE LABELS
    # --------------------------------------------------------

    R_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{R_kohm:.0f} kΩ</div>'

    C_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{C_nF:.0f} nF</div>'

    T_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{T:.0f} K</div>'

    # --------------------------------------------------------
    # INPUT PSD
    # --------------------------------------------------------

    S_in = 2.0 * kB * T * R

    # --------------------------------------------------------
    # RC POWER RESPONSE
    # --------------------------------------------------------

    H2 = 1.0 / (
        1.0 + (omega * R * C)**2
    )

    # --------------------------------------------------------
    # OUTPUT PSD
    # --------------------------------------------------------

    S_out = S_in * H2

    # --------------------------------------------------------
    # CUTOFF FREQUENCY
    # --------------------------------------------------------

    fc = 1.0 / (
        2.0 * np.pi * R * C
    )

    # --------------------------------------------------------
    # CUMULATIVE NOISE POWER FRACTION
    # --------------------------------------------------------

    cumulative_fraction = (
        2.0 / np.pi
    ) * np.arctan(
        omega * R * C
    )

    # --------------------------------------------------------
    # TOTAL OUTPUT NOISE
    # --------------------------------------------------------

    theoretical_power = kB * T / C

    theoretical_rms = np.sqrt(
        theoretical_power
    )

    # ========================================================
    # UPDATE EXISTING GRAPHICAL OBJECTS
    # ========================================================

    line_input.set_ydata(
        S_in * np.ones_like(f)
    )

    line_output.set_ydata(
        S_out
    )

    cutoff_line.set_xdata(
        [fc, fc]
    )

    line_cumulative.set_ydata(
        cumulative_fraction
    )

    cutoff_line_3.set_xdata(
        [fc, fc]
    )

    # ========================================================
    # UPDATE NUMERICAL OUTPUT
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial;
        font-size:15px;
        line-height:1.45;
        width:930px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Input thermal-noise PSD:</b>
    2kTR = {S_in:.4e} V²s

    &nbsp;&nbsp;&nbsp;

    <b>RC cutoff:</b>
    f<sub>c</sub> = {fc:.2f} Hz

    <br>

    <b>Total output mean-square noise:</b>
    kT/C = {theoretical_power:.4e} V²

    &nbsp;&nbsp;&nbsp;

    <b>RMS noise voltage:</b>
    {theoretical_rms:.4e} V

    </div>
    """

    # ========================================================
    # REDRAW ONLY THE EXISTING CANVASES
    # ========================================================

    fig1.canvas.draw_idle()
    fig2.canvas.draw_idle()
    fig3.canvas.draw_idle()

# ============================================================
# CONNECT SLIDERS TO THE UPDATE FUNCTION
# ============================================================

R_slider.observe(
    update_plots,
    names='value'
)

C_slider.observe(
    update_plots,
    names='value'
)

T_slider.observe(
    update_plots,
    names='value'
)

# ============================================================
# INITIAL NUMERICAL OUTPUT
# ============================================================

update_plots()

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial;
    font-size:15px;
    line-height:1.45;
    width:1050px;
    padding:11px 15px;
    border:1px solid #dfcdb9;
    background:#fffaf4;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#8b4d13;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The resistor produces an approximately flat thermal-noise spectrum, while the RC system suppresses its high-frequency components.
</div>

<div style="margin-bottom:4px;">
Increasing C lowers the cutoff frequency and reduces the total output noise power.
</div>

<div style="margin-bottom:4px;">
Changing R modifies both the input thermal-noise PSD and the RC cutoff frequency.
</div>

<div>
After integration over the complete spectrum, the total mean-square output noise is kT/C and is therefore independent of R.
</div>

</div>
""")

# ============================================================
# GRAPH LAYOUT
# ============================================================

top_graphs = HBox(
    [
        fig1.canvas,
        fig2.canvas
    ],
    layout=Layout(
        width='1050px',
        align_items='flex-start',
        justify_content='space-between',
        overflow='hidden'
    )
)

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        controls_card,
        top_graphs,
        fig3.canvas,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)